# Semana 44 · ¿Qué cambia cuando algo va muy rápido?

Este cuaderno acompaña a la sesión de proyecto de la semana 44: el factor de Lorentz, una lluvia de muones, los relojes del GPS y la energía relativista.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Calculadora del factor de Lorentz

$$\gamma = \frac{1}{\sqrt{1 - \beta^2}}.$$

Para velocidades muy pequeñas, $1 - \beta^2$ es tan parecido a 1 que el ordenador pierde precisión al restar. Por eso, además de $\gamma$, calculamos $\gamma - 1$ con la aproximación binomial cuando $\beta$ es pequeño.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

c = 299_792_458.0

def gamma(v):
    b = np.asarray(v, float) / c
    return 1 / np.sqrt(1 - b**2)

def gamma_menos_1(v):
    b = np.asarray(v, float) / c
    return np.where(b < 1e-3, 0.5 * b**2 + 3 / 8 * b**4, gamma(v) - 1)

objetos = [("persona andando", 1.4), ("avión de pasajeros", 250), ("Estación Espacial Internacional", 7_660),
           ("satélite GPS", 3_870), ("la Tierra alrededor del Sol", 29_780), ("viento solar rápido", 800e3),
           ("la sonda Parker en su máximo", 192e3), ("electrón de un tubo de televisor antiguo", 0.3 * c),
           ("muon de rayo cósmico", 0.9995 * c), ("protón del LHC", (1 - 9.5e-9) * c)]
print(f"{'objeto':<42}{'v/c':>12}{'γ − 1':>12}")
for nombre, v in objetos:
    print(f"{nombre:<42}{v / c:>12.3e}{float(gamma_menos_1(v)):>12.3e}")

In [ ]:
beta = np.linspace(0, 0.999, 1000)
plt.figure(figsize=(8, 4.5))
plt.plot(beta, 1 / np.sqrt(1 - beta**2), label="γ exacto")
plt.plot(beta, 1 + 0.5 * beta**2, "--", label="1 + β²/2 (aproximación)")
plt.ylim(0.9, 10); plt.xlabel("β = v/c"); plt.ylabel("γ"); plt.legend(); plt.grid(alpha=0.3)
plt.title("El factor de Lorentz"); plt.show()

Hasta $\beta \approx 0{,}3$, $\gamma$ apenas se separa de 1 y la aproximación es excelente. A partir de 0,9 se dispara.

## 2. Una lluvia de muones

Simulamos 100 000 muones creados a 15 km de altura que bajan en vertical a $\beta = 0{,}9995$. Cada uno vive un tiempo propio aleatorio con distribución exponencial de media 2,2 µs (la desintegración, como la radiactividad, es un proceso exponencial, semana 26). Contamos cuántos llegan al suelo con y sin dilatación del tiempo. (Simplificamos mucho: los muones reales nacen a distintas alturas, con distintas energías, y pierden energía al atravesar el aire.)

In [ ]:
rng = np.random.default_rng(44)
n, altura, beta_mu, tau = 100_000, 15_000.0, 0.9995, 2.197e-6
v = beta_mu * c
t_propio = rng.exponential(tau, n)

sin_rel = (v * t_propio > altura).mean()
con_rel = (v * gamma(v) * t_propio > altura).mean()
print(f"γ = {gamma(v):.1f}")
print(f"llegan al suelo sin relatividad: {100 * sin_rel:.2e} %  (exp(-h/(v τ)) = {100 * np.exp(-altura / (v * tau)):.2e} %)")
print(f"llegan al suelo con relatividad: {100 * con_rel:.1f} %  (exp(-h/(γ v τ)) = {100 * np.exp(-altura / (gamma(v) * v * tau)):.1f} %)")

h = np.linspace(0, altura, 200)
plt.figure(figsize=(8, 4.5))
plt.plot(h / 1000, np.exp(-h / (v * tau)), label="sin dilatación del tiempo")
plt.plot(h / 1000, np.exp(-h / (gamma(v) * v * tau)), label="con dilatación del tiempo")
plt.xlabel("distancia recorrida (km)"); plt.ylabel("fracción de muones que sobreviven"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

Sin relatividad, de cada millón de muones no llegaría prácticamente ninguno. Con ella, llega una buena parte. Desde el punto de vista del muon, el razonamiento es otro pero da lo mismo: para él, su vida dura 2,2 µs, pero los 15 km de atmósfera se contraen a $15/\gamma \approx 0{,}5$ km.

## 3. Los relojes del GPS

La dilatación por la velocidad (relatividad especial) y el efecto de la gravedad (relatividad general, semana 45) en los relojes de los satélites. Para la segunda usamos el resultado de la relatividad general en campo débil: un reloj más arriba en un campo gravitatorio adelanta en $\Delta\Phi/c^2$, donde $\Phi = -GM/r$ es el potencial gravitatorio.

In [ ]:
GM = 3.986004e14           # m³/s², para la Tierra
R_T = 6.371e6
r_gps = 26_560e3           # radio de la órbita GPS
v_gps = np.sqrt(GM / r_gps)
dia = 86_400

especial = -float(gamma_menos_1(v_gps)) * dia          # el reloj del satélite atrasa
general = (GM / R_T - GM / r_gps) / c**2 * dia          # y adelanta por estar más arriba
print(f"velocidad del satélite: {v_gps / 1000:.2f} km/s")
print(f"relatividad especial: {especial * 1e6:+.1f} µs al día")
print(f"relatividad general:  {general * 1e6:+.1f} µs al día")
print(f"total:                {(especial + general) * 1e6:+.1f} µs al día → {abs(especial + general) * c / 1000:.0f} km de error al día si no se corrigiera")

## 4. Energía cinética: Newton frente a Einstein

$$K_\text{Newton} = \tfrac{1}{2}mv^2, \qquad K_\text{Einstein} = (\gamma - 1)\,mc^2.$$

In [ ]:
beta = np.linspace(0.001, 0.99, 500)
K_newton = 0.5 * beta**2
K_einstein = 1 / np.sqrt(1 - beta**2) - 1
plt.figure(figsize=(8, 4.5))
plt.plot(beta, K_newton, label="Newton: ½mv²")
plt.plot(beta, K_einstein, label="Einstein: (γ − 1)mc²")
plt.ylim(0, 5); plt.xlabel("β"); plt.ylabel("energía cinética (en unidades de mc²)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
for b in [0.1, 0.5, 0.9]:
    print(f"β = {b}: Einstein/Newton = {(1 / np.sqrt(1 - b**2) - 1) / (0.5 * b**2):.3f}")

A velocidades pequeñas coinciden; cerca de $c$, la energía relativista crece sin límite y la de Newton se queda en $\tfrac{1}{2}mc^2$. Por eso los aceleradores necesitan cada vez más energía para ganar velocidades cada vez más pequeñas.

## 5. Para ir más lejos: sumar velocidades

$$u = \frac{u' + v}{1 + u'v/c^2}.$$

In [ ]:
def suma_relativista(u1, v):
    return (u1 + v) / (1 + u1 * v)          # en unidades de c

for u1, v in [(0.5, 0.5), (0.9, 0.9), (0.99, 0.99), (1.0, 0.5), (1e-6, 1e-6)]:
    print(f"{u1} c ⊕ {v} c = {suma_relativista(u1, v):.10f} c   (Galileo: {u1 + v} c)")

v = np.linspace(0, 1, 200)
plt.figure(figsize=(7, 4.5))
for u1 in [0.2, 0.5, 0.8, 0.95]:
    plt.plot(v, suma_relativista(u1, v), label=f"u' = {u1}c")
plt.axhline(1, color="k", ls="--", lw=0.8)
plt.xlabel("velocidad del tren v/c"); plt.ylabel("velocidad resultante u/c"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 6. Experimenta

1. En la lluvia de muones, ¿qué velocidad mínima necesita un muon para que llegue al suelo al menos el 10 %?
2. ¿A qué altura tendría que estar la órbita de un satélite para que los efectos especial y general se anularan exactamente?
3. Calcula cuánta energía haría falta para llevar una nave de 1000 kg a $0{,}1c$, $0{,}5c$ y $0{,}9c$, y compárala con el consumo de energía mundial de un año (unos $6\times10^{20}$ J).

In [ ]:
# Tu experimento aquí